# Tutorial: Policy Corpus Builder v0.1 Local File Walkthrough

This notebook shows the smallest useful `policy-corpus-builder` workflow:

- load a validated config
- resolve queries
- read local fixture-backed records with the `local-file` adapter
- normalize them into `NormalizedDocument` records
- deduplicate deterministically
- export the final corpus to JSONL
- inspect the output


## Setup

This walkthrough uses the bundled example config and fixture data in the repository.

In [ ]:
from __future__ import annotations

from pathlib import Path

from policy_corpus_builder.adapters import get_adapter
from policy_corpus_builder.config import load_and_validate_config
from policy_corpus_builder.exporters.jsonl import export_documents_jsonl
from policy_corpus_builder.pipeline import normalize_adapter_results
from policy_corpus_builder.postprocess import deduplicate_documents
from policy_corpus_builder.queries import load_queries

repo_root = Path.cwd().resolve()
while not (repo_root / "pyproject.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

config_path = repo_root / "examples" / "local_file.toml"
output_dir = repo_root / "examples" / "outputs" / "notebook-demo"

config_path, output_dir


## Load And Validate The Config

The config describes the query source, source adapter, deduplication fields, and export format.

In [ ]:
config = load_and_validate_config(config_path)

{
    "project": config.project.name,
    "query_inventory": config.queries.inventory,
    "sources": [source.name for source in config.sources],
    "deduplicate_fields": list(config.normalization.deduplicate_fields),
    "export_formats": list(config.export.formats),
}


## Resolve Queries

Queries can come from an inventory file or inline config items. The loader normalizes both into one internal representation.

In [ ]:
queries = load_queries(config, base_path=config_path.parent)

[(query.query_id, query.text, query.origin) for query in queries]


## Run The Local-File Adapter

The `local-file` adapter reads the fixture once and then collects matching records for each query.

In [ ]:
source = config.sources[0]
adapter = get_adapter(source.adapter)
loaded_source = adapter.load_source(source, base_path=config_path.parent)

len(loaded_source), sorted(loaded_source[0].keys())


## Normalize Records Into Shared Documents

Each adapter result is converted into a `NormalizedDocument`, and the active query is attached during normalization.

In [ ]:
documents = []
raw_result_count = 0

for query in queries:
    raw_results = adapter.collect(
        source,
        query,
        base_path=config_path.parent,
        loaded_source=loaded_source,
    )
    raw_result_count += len(raw_results)
    documents.extend(
        normalize_adapter_results(raw_results, source=source, query=query)
    )

raw_result_count, len(documents), documents[0].to_dict()


## Deduplicate Deterministically

Deduplication uses the configured normalized metadata fields and keeps the first document seen for each key.

In [ ]:
deduped = deduplicate_documents(tuple(documents), config=config.normalization)

{
    "raw_normalized_documents": len(documents),
    "final_documents": len(deduped.documents),
    "duplicates_removed": deduped.duplicates_removed,
}


## Export To JSONL And Inspect The Result

The exporter writes one normalized record per line.

In [ ]:
output_path = export_documents_jsonl(deduped.documents, output_dir=output_dir)
lines = output_path.read_text(encoding="utf-8").splitlines()

{
    "output_path": str(output_path),
    "record_count": len(lines),
    "first_record": lines[0] if lines else None,
}


## Next Step

Once this workflow feels comfortable, the next move is usually to replace the fixture with your own structured local source file and keep the same normalization, deduplication, and export steps.